# Desafío: Evaluando la preparación de datos para aprendizaje de máquina supervisado

## Caso: aprobación de créditos para microemprendedores

# Hector Lopez 
## Data Science

## 1. Importación de librerías

Se utilizan herramientas de `pandas` para manipular los datos y componentes de `scikit-learn` para el preprocesamiento, modelado y validación.

In [2]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda valor: f"{valor:,.4f}")

## 2. Carga del dataset

El código busca el archivo en el mismo directorio del notebook y también en `/mnt/data`, lo que facilita su ejecución tanto en Jupyter como en Google Colab.

In [2]:
import pandas as pd

# Ruta del archivo Excel
ruta_dataset = r"C:\Users\HP\OneDrive\Desktop\Especializacion\Aprendizaje de máquina supervisado\03. Unidad 3 Evaluación y optimización de modelos\09. Apoyo prueba - optimizacion_modelos_predictivos.xlsx"

# Cargar el dataset
df = pd.read_excel(ruta_dataset)

# Mostrar información básica
print("✅ Dataset cargado correctamente")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

# Mostrar las primeras filas
df.head()

✅ Dataset cargado correctamente
Filas: 500
Columnas: 7


,edad,nivel_participacion,tiempo_en_plataforma,tipo_inscripcion,region,completo,cursos_futuros
0,19,0.63,9.3,Premium,Norte,1,4
1,53,0.57,9.7,Libre,Centro,0,2
2,58,0.73,1.1,Premium,Norte,1,4
3,27,0.55,8.0,Libre,Centro,0,4
4,58,0.25,5.5,Libre,Sur,0,1


## 3. Inspección y comprensión de los datos

Se revisan:

- nombres y tipos de columnas;
- valores faltantes;
- estadísticas descriptivas;
- cantidad de categorías;
- distribución de la variable objetivo.

In [3]:
print("Información general del dataset:")
df.info()

print("\nValores nulos por columna:")
display(df.isna().sum().to_frame("valores_nulos"))

print("\nFilas duplicadas:", df.duplicated().sum())

Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   edad                  500 non-null    int64  
 1   nivel_participacion   500 non-null    float64
 2   tiempo_en_plataforma  500 non-null    float64
 3   tipo_inscripcion      500 non-null    str    
 4   region                500 non-null    str    
 5   completo              500 non-null    int64  
 6   cursos_futuros        500 non-null    int64  
dtypes: float64(2), int64(3), str(2)
memory usage: 27.5 KB

Valores nulos por columna:


,valores_nulos
edad,0
nivel_participacion,0
tiempo_en_plataforma,0
tipo_inscripcion,0
region,0
completo,0
cursos_futuros,0



Filas duplicadas: 0


In [4]:
print("Estadísticas de las variables numéricas:")
display(df.describe().T)

print("Frecuencias de las variables categóricas:")
for columna in ["region", "genero"]:
    print(f"\n{columna.upper()}")
    if columna in df.columns:
        display(df[columna].value_counts(dropna=False).to_frame("frecuencia"))
    else:
        print(f"La columna '{columna}' no existe en el dataset. Revisa el archivo CSV y el separador.")

Estadísticas de las variables numéricas:


,count,mean,std,min,25%,50%,75%,max
edad,500.0,39.04800,12.280689,18.0,29.0000,40.00,49.00,59.00
nivel_participacion,500.0,0.55726,0.234569,0.1,0.3875,0.57,0.73,0.99
tiempo_en_plataforma,500.0,8.62660,5.062475,1.0,4.6000,8.50,12.40,23.50
completo,500.0,0.47200,0.499715,0.0,0.0000,0.00,1.00,1.00
cursos_futuros,500.0,3.58800,2.008563,0.0,2.0000,4.00,5.00,8.00


Frecuencias de las variables categóricas:

REGION


,frecuencia
region,
Centro,211
Norte,161
Sur,128



GENERO
La columna 'genero' no existe en el dataset. Revisa el archivo CSV y el separador.


In [ ]:
# El archivo CSV tiene columnas separadas por ';'. Si se leyó con el separador
# por defecto (',') quedó todo en una sola columna, y por eso no aparece
# 'credito_aprobado' en 'df'.


# Validación explícita de la columna objetivo
if "credito_aprobado" not in df.columns:
    if "credito_aprobado" not in df.columns:
        columnas_esperadas = [
            "edad",
            "ingreso_mensual",
            "deuda_actual",
            "region",
            "genero",
            "credito_aprobado",
        ]
        raise KeyError(
            "La columna 'credito_aprobado' no existe en el dataset cargado. "
            "El DataFrame que se está usando no corresponde al dataset de aprobación "
            "de créditos esperado por este notebook. "
            "Revisa que `ruta_dataset` apunte al archivo correcto y que, si es CSV, "
            "se cargue con `sep=';'` (y no con el separador por defecto ','). "
            f"Se esperaban estas columnas: {columnas_esperadas}. "
            f"Columnas disponibles en `df`: {list(df.columns)}"
        )

distribucion_objetivo = pd.DataFrame({
    "cantidad": df["credito_aprobado"].value_counts().sort_index(),
    "proporcion": df["credito_aprobado"].value_counts(normalize=True).sort_index()
})

print("Distribución de la variable objetivo (0 = rechazado, 1 = aprobado):")
display(distribucion_objetivo)

KeyError: "La columna 'credito_aprobado' no existe en el dataset cargado. Revisa que el CSV correcto esté apuntado en `ruta_dataset` y que use ';' como separador. Columnas disponibles: ['edad', 'nivel_participacion', 'tiempo_en_plataforma', 'tipo_inscripcion', 'region', 'completo', 'cursos_futuros']"

### Identificación de variables

**Variables numéricas**

- `edad`
- `ingreso_mensual`
- `deuda_actual`

**Variables categóricas nominales**

- `region`
- `genero`

**Variable objetivo**

- `credito_aprobado`

La variable objetivo toma dos categorías: `0` y `1`.

## 4. Identificación del tipo de aprendizaje

Este problema corresponde a **aprendizaje supervisado** porque el dataset histórico incluye tanto las características de cada postulante como la respuesta conocida `credito_aprobado`. El algoritmo puede aprender la relación entre las variables predictoras y dicha etiqueta.

Es una **clasificación binaria**, no de regresión, porque la variable objetivo es categórica y tiene dos clases:

- `0`: crédito no aprobado.
- `1`: crédito aprobado.

Una regresión se utilizaría si el objetivo fuera un valor numérico continuo, por ejemplo, el monto máximo de crédito que puede recibir cada persona.

## 5. Separación de variables predictoras y objetivo

La separación se realiza **antes** de ajustar el pipeline, como buena práctica de Machine Learning.

In [6]:
X = df.drop(columns="credito_aprobado")
y = df["credito_aprobado"]

variables_numericas = ["edad", "ingreso_mensual", "deuda_actual"]
variables_categoricas = ["region", "genero"]

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)
print("\nVariables numéricas:", variables_numericas)
print("Variables categóricas:", variables_categoricas)

Dimensión de X: (500, 5)
Dimensión de y: (500,)

Variables numéricas: ['edad', 'ingreso_mensual', 'deuda_actual']
Variables categóricas: ['region', 'genero']


## 6. Decisiones de preprocesamiento

### Codificación categórica: `OneHotEncoder`

`region` y `genero` son variables nominales: sus categorías no tienen un orden matemático natural. Por eso se utiliza `OneHotEncoder`, que crea columnas binarias y evita imponer relaciones artificiales como “Metropolitana > Biobío”.

Se configura `handle_unknown="ignore"` para que el pipeline pueda procesar categorías nuevas durante una predicción futura sin producir un error.

### Escalamiento numérico: `StandardScaler`

Se utiliza `StandardScaler` porque `edad`, `ingreso_mensual` y `deuda_actual` están expresadas en escalas muy diferentes. El escalador centra cada variable alrededor de cero y la expresa en unidades de desviación estándar.

Esto favorece la Regresión Logística porque:

- evita que las variables con valores monetarios grandes dominen la optimización;
- facilita la convergencia del algoritmo;
- permite que la regularización trate las características de manera comparable.

El escalamiento se incorpora dentro del pipeline para que sus parámetros se calculen exclusivamente con los datos de entrenamiento de cada partición, evitando **data leakage**.

## 7. Construcción de `ColumnTransformer` y `Pipeline`

El `ColumnTransformer` aplica transformaciones distintas según el tipo de variable. Después, el `Pipeline` conecta el preprocesamiento con el modelo de Regresión Logística.

In [7]:
preprocesamiento = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), variables_numericas),
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categoricas,
        ),
    ],
    remainder="drop",
)

modelo = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

pipeline_crediticio = Pipeline(
    steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", modelo),
    ]
)

pipeline_crediticio

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float

## 8. Validación cruzada

Se utiliza `StratifiedKFold` con cinco particiones porque es una tarea de clasificación. La estratificación conserva aproximadamente la proporción de créditos aprobados y rechazados en cada fold.

Las métricas solicitadas son:

- **Accuracy:** proporción total de predicciones correctas.
- **F1 macro:** calcula el F1 de cada clase y luego les asigna el mismo peso, independientemente de su frecuencia.

In [8]:
cv_estratificada = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

metricas = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
}

resultados_cv = cross_validate(
    estimator=pipeline_crediticio,
    X=X,
    y=y,
    cv=cv_estratificada,
    scoring=metricas,
    return_train_score=True,
    n_jobs=-1,
)

resultados_por_fold = pd.DataFrame({
    "fold": range(1, cv_estratificada.get_n_splits() + 1),
    "accuracy_validacion": resultados_cv["test_accuracy"],
    "f1_macro_validacion": resultados_cv["test_f1_macro"],
    "accuracy_entrenamiento": resultados_cv["train_accuracy"],
    "f1_macro_entrenamiento": resultados_cv["train_f1_macro"],
})

display(resultados_por_fold)

,fold,accuracy_validacion,f1_macro_validacion,accuracy_entrenamiento,f1_macro_entrenamiento
0,1,0.9300,0.9261,0.9575,0.9547
1,2,0.9200,0.9121,0.9550,0.9522
2,3,0.9400,0.9363,0.9475,0.9439
3,4,0.9700,0.9689,0.9525,0.9494
4,5,0.9200,0.9132,0.9425,0.9389


In [9]:
resumen_metricas = pd.DataFrame({
    "métrica": ["Accuracy", "F1 macro"],
    "promedio_validación": [
        resultados_cv["test_accuracy"].mean(),
        resultados_cv["test_f1_macro"].mean(),
    ],
    "desviación_estándar": [
        resultados_cv["test_accuracy"].std(),
        resultados_cv["test_f1_macro"].std(),
    ],
})

display(resumen_metricas)

accuracy_promedio = resultados_cv["test_accuracy"].mean()
f1_macro_promedio = resultados_cv["test_f1_macro"].mean()

print(f"Accuracy promedio: {accuracy_promedio:.4f} ({accuracy_promedio:.2%})")
print(f"F1 macro promedio: {f1_macro_promedio:.4f} ({f1_macro_promedio:.2%})")

,métrica,promedio_validación,desviación_estándar
0,Accuracy,0.9360,0.0185
1,F1 macro,0.9313,0.0208


Accuracy promedio: 0.9360 (93.60%)
F1 macro promedio: 0.9313 (93.13%)


## 9. Interpretación crítica de las métricas

La distribución de la variable objetivo presenta una diferencia moderada entre las clases, con más créditos aprobados que rechazados. En este contexto, **accuracy** sigue siendo útil porque muestra el porcentaje global de aciertos, pero puede favorecer la clase más frecuente.

Por esta razón, considero que **F1 macro es la métrica más representativa** para este desafío. Esta métrica evalúa por separado la capacidad del modelo para reconocer créditos aprobados y rechazados y después promedia ambos resultados con el mismo peso. Así, un buen resultado no puede depender únicamente de predecir correctamente la clase mayoritaria.

En un sistema crediticio, clasificar incorrectamente cualquiera de las dos clases puede tener consecuencias:

- aprobar a una persona de alto riesgo puede generar pérdidas;
- rechazar a una persona viable puede hacer perder un buen cliente.

Por ello, F1 macro entrega una evaluación más equilibrada del comportamiento general del clasificador. En un proyecto real también sería recomendable revisar la matriz de confusión, `precision`, `recall` de cada clase y definir cuál tipo de error tiene mayor costo para la fintech.

## 10. ¿Qué ocurriría si no se escalan los datos?

La Regresión Logística podría entrenarse sin escalamiento, pero las variables monetarias (`ingreso_mensual` y `deuda_actual`) tienen magnitudes muy superiores a `edad`. Esto puede:

- volver menos eficiente el proceso de optimización;
- dificultar la convergencia;
- hacer que la regularización penalice de manera desigual los coeficientes;
- reducir la estabilidad numérica del entrenamiento.

El escalamiento no cambia el significado de las observaciones, sino que coloca las variables numéricas en una escala comparable.

Además, hacer el escalamiento antes de la validación cruzada sería un error, porque utilizaría información estadística de todos los registros. Al mantenerlo dentro del pipeline, cada fold calcula su media y desviación estándar únicamente con los datos de entrenamiento.

## 11. Ajuste final del pipeline

Después de evaluar el enfoque mediante validación cruzada, se puede entrenar el pipeline con todo el dataset para dejarlo preparado para predicciones futuras.

In [10]:
pipeline_crediticio.fit(X, y)

print("Pipeline final entrenado correctamente con los 500 registros.")
print("Clases del modelo:", pipeline_crediticio.named_steps["modelo"].classes_)

Pipeline final entrenado correctamente con los 500 registros.
Clases del modelo: [0 1]


In [11]:
# Ejemplo ilustrativo de predicción con una persona nueva
nuevo_postulante = pd.DataFrame({
    "edad": [35],
    "ingreso_mensual": [750000],
    "deuda_actual": [180000],
    "region": ["Metropolitana"],
    "genero": ["Femenino"],
})

prediccion = pipeline_crediticio.predict(nuevo_postulante)[0]
probabilidad = pipeline_crediticio.predict_proba(nuevo_postulante)[0, 1]

print("Predicción:", "Aprobado" if prediccion == 1 else "No aprobado")
print(f"Probabilidad estimada de aprobación: {probabilidad:.2%}")

Predicción: Aprobado
Probabilidad estimada de aprobación: 96.65%


## 12. Conclusiones

- El problema fue identificado correctamente como aprendizaje supervisado de clasificación binaria.
- Las variables categóricas se transformaron con `OneHotEncoder`.
- Las variables numéricas se estandarizaron con `StandardScaler`.
- `ColumnTransformer` permitió aplicar tratamientos diferentes a cada grupo de columnas.
- El preprocesamiento y la Regresión Logística se integraron en un único pipeline reproducible.
- La validación cruzada estratificada proporcionó una estimación más robusta que una sola división de entrenamiento y prueba.
- `F1 macro` se considera la métrica principal porque evalúa ambas clases con la misma importancia.
- Mantener las transformaciones dentro del pipeline evita fuga de información y permite aplicar exactamente el mismo proceso a nuevos postulantes.